# 04 — Visualization

Generate publication-ready charts from the master DUI-by-state dataset.

Charts produced:
1. **Choropleth maps** (twitter_landscape) — fatality rate per 100M VMT, felony status, IID status, max speed limit, prior DWI %
2. **Scatter** (twitter_landscape) — alcohol consumption vs fatality rate, colored by region
3. **Ranked bars** (instagram_portrait) — top/bottom 10 states by fatality rate per VMT
4. **IID comparison** (twitter_landscape) — IID vs non-IID mean fatality rates

All outputs saved to `outputs/` with @unwelcomedata watermark.

In [ ]:
import sys
import os
from pathlib import Path

%matplotlib inline
import pandas as pd
import yaml

# Find project root by walking up from cwd until we find config.yaml
PROJECT = Path.cwd()
while not (PROJECT / "config.yaml").exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

from src.viz import (
    choropleth_map,
    scatter_chart,
    ranked_bar_chart,
    comparison_chart,
    line_chart,
    save_chart,
)

# Load config
with open(PROJECT / "config.yaml") as f:
    cfg = yaml.safe_load(f)

# Load master table — use v4 export + supplement from DuckDB for exploration
df = pd.read_parquet(PROJECT / "export" / "dui_by_state_v4.parquet")

# DuckDB connection (read-only) for cells that query interim tables
import duckdb
con = duckdb.connect(str(PROJECT / "data" / "project.duckdb"), read_only=True)

# Add back columns from DuckDB for exploratory charts (these exist in DB but not in v4 export)
try:
    extras = con.execute('''
        SELECT s.state_fips, s.lat, s.lng,
               v.max_speed_limit_mph, v.vmt_millions_2022,
               ROUND(s.traffic_fatalities_2024 * 100.0 / (v.vmt_millions_2022 / 100.0), 2) AS total_fatality_rate_per_100m_vmt,
               ROUND(n.alcohol_impaired_fatalities_2024 * 100.0 / (v.vmt_millions_2022 / 100.0), 2) AS alcohol_fatality_rate_per_100m_vmt,
               ps.pct_impaired_with_prior_dwi, ps.pct_all_with_prior_dwi,
               ps.median_crash_speed_limit, ps.pct_crashes_high_speed,
               ps.impaired_drivers_known_history, ps.impaired_with_prior_dwi,
               ac.ethanol_per_capita_gallons_2022,
               da.total_dui_arrests, da.reporting_agencies, da.reporting_population,
               ROUND(da.total_dui_arrests * 100000.0 / da.reporting_population, 1) AS dui_arrest_rate_per_100k_reporting,
               bt.total_drivers AS total_drivers_in_fatal_crashes
        FROM states s
        LEFT JOIN speed_limits_vmt v ON s.state_name = v.state_name
        LEFT JOIN nhtsa_imputed_2024 n ON s.state_name = n.state_name
        LEFT JOIN fars_prior_dwi_speed ps ON s.state_fips = ps.state_fips
        LEFT JOIN alcohol_consumption ac ON s.state_fips = ac.state_fips
        LEFT JOIN dui_arrests_2023 da ON s.state_fips = da.state_fips
        LEFT JOIN fars_bac_testing_2024 bt ON s.state_fips = bt.state_fips
        ORDER BY s.state_fips
    ''').fetchdf()
    extras['state_fips'] = extras['state_fips'].astype(str).str.zfill(2)
    df = df.merge(extras, on='state_fips', how='left', suffixes=('', '_dup'))
    df = df[[c for c in df.columns if not c.endswith('_dup')]]
    print(f'  (supplemented with {len(extras.columns)-1} exploration columns from DuckDB)')
except Exception as e:
    print(f'  Note: could not load exploration extras: {e}')

print(f"Project root: {PROJECT}")
print(f"Master table: {df.shape[0]} states x {df.shape[1]} columns")
df.head(3)

## 1. Choropleth Maps

Flexible map function — swap `column` and `mode` to explore any measure.

In [ ]:
# --- Map 1: Alcohol fatality rate per 100M VMT (heat) ---

fig = choropleth_map(
    df,
    column="alcohol_fatality_rate_per_100m_vmt",
    title="Alcohol-Impaired Fatality Rate by State",
    subtitle="Deaths per 100 million vehicle miles traveled (2024)",
    source="NHTSA FARS 2024, FHWA VMT 2022",
    mode="heat",
    legend_title="Deaths per 100M VMT",
    preset="twitter_landscape",
    annotate=True,
)
save_chart(fig, cfg, "map_alcohol_fatality_rate_vmt", preset="twitter_landscape", add_watermark="@unwelcomedata", close=False)
fig

In [ ]:
# --- Map 2: Felony vs misdemeanor (category) ---

# Create a clean felony label column
df["felony_label"] = df["first_offense_felony"].map({1.0: "Can be felony", 0.0: "Always misdemeanor"})
df.loc[df["felony_label"].isna(), "felony_label"] = "Unknown"

fig = choropleth_map(
    df,
    column="felony_label",
    title="First-Offense DUI: Felony Possible?",
    subtitle="States where first DUI can be charged as felony (e.g. with child in car or injury)",
    source="NCSL DUI/DWI criminal status laws",
    mode="category",
    category_colors={"Can be felony": "#DC2626", "Always misdemeanor": "#2563EB", "Unknown": "#E5E7EB"},
    legend_title="First-offense status",
    preset="twitter_landscape",
)
save_chart(fig, cfg, "map_felony_status", preset="twitter_landscape", add_watermark="@unwelcomedata", close=False)
fig

In [ ]:
# --- Map 3: IID requirement (category) ---

df["iid_label"] = df["iid_all_offender"].map({1: "All offenders", 0: "Repeat/high-BAC only"})

fig = choropleth_map(
    df,
    column="iid_label",
    title="Ignition Interlock: All First Offenders?",
    subtitle="States requiring IID for all DUI offenders vs repeat/high-BAC only",
    source="IIHS, GHSA, NHTSA enforcement compilations",
    mode="category",
    category_colors={"All offenders": "#16A34A", "Repeat/high-BAC only": "#D97706"},
    legend_title="IID requirement",
    preset="twitter_landscape",
)
save_chart(fig, cfg, "map_iid_status", preset="twitter_landscape", add_watermark="@unwelcomedata", close=False)
fig

In [ ]:
# --- Map 4: Max speed limit (heat) ---

fig = choropleth_map(
    df,
    column="max_speed_limit_mph",
    title="Maximum Posted Speed Limit by State",
    subtitle="Rural interstate max speed (mph)",
    source="IIHS, August 2026",
    mode="heat",
    cmap=["#DBEAFE", "#60A5FA", "#2563EB", "#7C3AED", "#4C1D95"],
    legend_title="Max speed (mph)",
    preset="twitter_landscape",
    annotate=True,
)
save_chart(fig, cfg, "map_max_speed_limit", preset="twitter_landscape", add_watermark="@unwelcomedata", close=False)
fig

In [ ]:
# --- Map 5: Prior DWI % (heat) ---

fig = choropleth_map(
    df,
    column="pct_impaired_with_prior_dwi",
    title="Repeat Offenders in Fatal Crashes",
    subtitle="% of impaired drivers in fatal crashes with a prior DWI conviction",
    source="NHTSA FARS 2024",
    mode="heat",
    cmap=["#FEF3C7", "#FBBF24", "#D97706", "#DC2626", "#7F1D1D"],
    legend_title="% with prior DWI",
    preset="twitter_landscape",
    annotate=True,
)
save_chart(fig, cfg, "map_prior_dwi_pct", preset="twitter_landscape", add_watermark="@unwelcomedata", close=False)
fig

## 2. Scatter — Consumption vs Fatality Rate

In [ ]:
fig = scatter_chart(
    df,
    x="ethanol_per_capita_gallons_2022",
    y="alcohol_fatality_rate_per_100m_vmt",
    color_by="region",
    title="Alcohol Consumption vs Impaired-Driving Deaths",
    subtitle="Each dot is a state. Per-VMT fatality rate controls for driving exposure.",
    source="NIAAA consumption 2022, NHTSA FARS 2024, FHWA VMT 2022",
    xlabel="Per capita ethanol (gallons, 2022)",
    ylabel="Alcohol fatalities per 100M VMT",
    annotate=True,
    preset="twitter_landscape",
)
save_chart(fig, cfg, "scatter_consumption_vs_fatality", preset="twitter_landscape", add_watermark="@unwelcomedata", close=False)
fig

## 3. Ranked Bars — Top/Bottom 10 by Fatality Rate (per VMT)

In [ ]:
fig = ranked_bar_chart(
    df,
    x="state_name",
    y="alcohol_fatality_rate_per_100m_vmt",
    title="Worst & Best States for Impaired-Driving Deaths",
    subtitle="Alcohol fatalities per 100M vehicle miles traveled (2024)",
    source="NHTSA FARS 2024, FHWA VMT 2022",
    top_n=10,
    bottom_n=10,
    value_fmt="{:.2f}",
    preset="instagram_portrait",
)
save_chart(fig, cfg, "ranked_top_bottom_10_vmt", preset="instagram_portrait", add_watermark="@unwelcomedata", close=False)
fig

## 4. IID vs Non-IID Comparison

In [ ]:
# Create IID group label
df["iid_group"] = df["iid_all_offender"].map({1: "IID for all offenders", 0: "No universal IID"})

fig = comparison_chart(
    df,
    group_col="iid_group",
    value_col="alcohol_fatality_rate_per_100m_vmt",
    title="Does Mandatory IID Reduce Impaired-Driving Deaths?",
    subtitle="Mean alcohol fatality rate per 100M VMT by IID policy (error bars = standard error)",
    source="NHTSA FARS 2024, FHWA VMT 2022, IIHS/GHSA enforcement data",
    colors={"IID for all offenders": "#16A34A", "No universal IID": "#DC2626"},
    value_fmt="{:.2f}",
    preset="twitter_landscape",
)
save_chart(fig, cfg, "comparison_iid_vs_no_iid", preset="twitter_landscape", add_watermark="@unwelcomedata", close=False)
fig

## Quick exploration

One-liner functions to explore any columns. Just call and view inline.

In [ ]:
def quick_map(column, title=None, mode="heat", **kwargs):
    """Choropleth any column. mode='heat' for numeric, 'category' for discrete."""
    title = title or column.replace('_', ' ').title()
    return choropleth_map(df, column=column, title=title, mode=mode, preset='twitter_landscape', annotate=True, **kwargs)


def quick_scatter(x, y, title=None, color_by='region', **kwargs):
    """Scatter any two numeric columns. Colored by region by default."""
    title = title or f"{x.replace('_',' ').title()} vs {y.replace('_',' ').title()}"
    return scatter_chart(df, x=x, y=y, color_by=color_by, title=title, preset='twitter_landscape', annotate=True, **kwargs)


def quick_bars(y, x='state_name', title=None, top_n=10, bottom_n=0, **kwargs):
    """Ranked bar chart. Shows top_n (and optionally bottom_n) states."""
    title = title or f"Top {top_n} States by {y.replace('_',' ').title()}"
    return ranked_bar_chart(df, x=x, y=y, title=title, top_n=top_n, bottom_n=bottom_n, preset='instagram_portrait', **kwargs)


def quick_compare(group_col, value_col, title=None, **kwargs):
    """Compare group means for any grouping column vs any numeric column."""
    title = title or f"{value_col.replace('_',' ').title()} by {group_col.replace('_',' ').title()}"
    return comparison_chart(df, group_col=group_col, value_col=value_col, title=title, preset='twitter_landscape', **kwargs)


def quick_trend(x='year', y='impaired_fatalities', data=None, title=None, **kwargs):
    """Line trend chart. Pass a different DataFrame via data= if needed."""
    title = title or f"{y.replace('_',' ').title()} over {x.replace('_',' ').title()}"
    src = data if data is not None else df
    return line_chart(src, x=x, y=y, title=title, preset='twitter_landscape', **kwargs)


def quick_bubble_map(color_col, size_col, title=None, **kwargs):
    """Map with state fill = color_col, bubble overlay = size_col."""
    title = title or f"{color_col.replace('_',' ').title()} (fill) + {size_col.replace('_',' ').title()} (bubble)"
    fig = choropleth_map(df, column=color_col, title=title, mode='heat', preset='twitter_landscape', annotate=False, **kwargs)
    # Overlay bubbles using lat/lng from the dataframe
    ax = fig.axes[0]
    import numpy as np
    from src.viz import PALETTE
    plot_df = df[['state_abbr', 'lat', 'lng', size_col]].dropna()
    sizes = (plot_df[size_col] - plot_df[size_col].min()) / (plot_df[size_col].max() - plot_df[size_col].min())
    # We need projected coords — use the geo data instead
    import geopandas as gpd
    from pathlib import Path
    shp = Path('data/raw/geo/cb_2022_us_state_20m.shp')
    geo = gpd.read_file(shp)
    geo = geo[~geo['STATEFP'].isin({'60','66','69','72','78'})].to_crs('EPSG:5070')
    geo['cx'] = geo.geometry.centroid.x
    geo['cy'] = geo.geometry.centroid.y
    merged = geo[['STUSPS','cx','cy']].merge(df[['state_abbr', size_col]], left_on='STUSPS', right_on='state_abbr')
    vals = merged[size_col].fillna(0)
    norm_sizes = 20 + 300 * (vals - vals.min()) / (vals.max() - vals.min())
    ax.scatter(merged['cx'], merged['cy'], s=norm_sizes, alpha=0.6, color='#E76F51', edgecolors='white', linewidths=0.5, zorder=5)
    return fig


def quick_bivariate_map(x_col, y_col, title=None, n_bins=3,
                         highlight_fips=None, highlight_label=None,
                         highlight_style=None):
    """
    Bivariate choropleth — 3x3 color grid showing two variables.
    
    Optional: highlight_fips = set of FIPS codes (2-digit strings) to outline.
    highlight_label = legend label for those states (e.g. 'Always misdemeanor').
    highlight_style = dict of plot kwargs (edgecolor, linewidth, linestyle).
    """
    import geopandas as gpd
    import numpy as np
    from pathlib import Path
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    from src.viz import PRESETS, STYLE, _fig_for_preset, _add_title_block, _shift_alaska_hawaii, _load_states_geo
    title = title or f"{x_col.replace('_',' ').title()} vs {y_col.replace('_',' ').title()}"
    # Default highlight style
    hl_style = {'edgecolor': 'black', 'linewidth': 2.5, 'linestyle': '--'}
    if highlight_style:
        hl_style.update(highlight_style)
    # 3x3 bivariate color grid
    biv_colors = [
        ['#E8E8E8', '#B8D6BE', '#2A9D8F'],  # y_low: x_low -> x_high
        ['#DFC27D', '#B5A068', '#6D8A6D'],  # y_mid
        ['#E76F51', '#C45A3C', '#264653'],  # y_high: x_low -> x_high
    ]
    data = df[['state_fips', x_col, y_col]].dropna().copy()
    data['state_fips'] = data['state_fips'].astype(str).str.zfill(2)
    data['_xbin'] = pd.qcut(data[x_col], n_bins, labels=False, duplicates='drop')
    data['_ybin'] = pd.qcut(data[y_col], n_bins, labels=False, duplicates='drop')
    data['_color'] = data.apply(lambda r: biv_colors[int(min(r['_ybin'], n_bins-1))][int(min(r['_xbin'], n_bins-1))], axis=1)
    geo = _load_states_geo()
    geo = _shift_alaska_hawaii(geo)
    geo = geo.merge(data[['state_fips','_color']], left_on='STATEFP', right_on='state_fips', how='left')
    geo['_color'] = geo['_color'].fillna('#E5E7EB')
    fig, ax = _fig_for_preset('twitter_landscape')
    ax.set_axis_off()
    for color in geo['_color'].unique():
        geo[geo['_color'] == color].plot(ax=ax, color=color, edgecolor='white', linewidth=0.5)
    # Highlight overlay: draw selected states with a distinct border
    if highlight_fips:
        hl_geo = geo[geo['STATEFP'].isin(highlight_fips)]
        hl_geo.plot(ax=ax, facecolor='none',
                    edgecolor=hl_style['edgecolor'],
                    linewidth=hl_style['linewidth'],
                    linestyle=hl_style['linestyle'],
                    zorder=4)
    # Legend grid
    legend_ax = fig.add_axes([0.82, 0.15, 0.12, 0.12])
    for yi in range(n_bins):
        for xi in range(n_bins):
            legend_ax.add_patch(mpatches.Rectangle((xi, yi), 1, 1, color=biv_colors[yi][xi]))
    legend_ax.set_xlim(0, n_bins)
    legend_ax.set_ylim(0, n_bins)
    legend_ax.set_xlabel(x_col.replace('_',' ').title()[:15] + ' →', fontsize=7)
    legend_ax.set_ylabel(y_col.replace('_',' ').title()[:15] + ' →', fontsize=7)
    legend_ax.set_xticks([])
    legend_ax.set_yticks([])
    # Add highlight legend entry if provided
    if highlight_fips and highlight_label:
        from matplotlib.lines import Line2D
        hl_handle = Line2D([0], [0], color=hl_style['edgecolor'],
                           linewidth=hl_style['linewidth'],
                           linestyle=hl_style['linestyle'],
                           label=highlight_label)
        ax.legend(handles=[hl_handle], loc='lower left', fontsize=9, framealpha=0.9)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left')
    fig.subplots_adjust(left=0.02, right=0.95, top=0.90, bottom=0.05)
    return fig


print('Quick functions ready: quick_map, quick_scatter, quick_bars, quick_compare, quick_trend, quick_bubble_map, quick_bivariate_map')

In [ ]:
def cols():
    """Print all columns in the master table, grouped by type."""
    numeric = [c for c in df.columns if df[c].dtype in ('float64', 'int64')]
    categorical = [c for c in df.columns if df[c].dtype == 'object']
    print(f'=== NUMERIC ({len(numeric)}) ===')
    for c in numeric:
        print(f'  {c}')
    print(f'\n=== CATEGORICAL ({len(categorical)}) ===')
    for c in categorical:
        vals = df[c].nunique()
        print(f'  {c}  ({vals} unique)')

cols()

In [ ]:
# --- Try it: consumption vs arrest rate ---
quick_scatter('ethanol_per_capita_gallons_2022', 'dui_arrest_rate_per_100k_reporting',
              title='Alcohol Consumption vs DUI Arrest Rate')

In [ ]:
# More examples (uncomment any):
# quick_map('dui_arrest_rate_per_100k', title='DUI Arrest Rate per 100k')
# quick_map('ethanol_per_capita_gallons_2022', title='Per Capita Alcohol Consumption')
# quick_scatter('dui_arrest_rate_per_100k', 'alcohol_fatality_rate_per_100m_vmt')
# quick_bars('dui_arrest_rate_per_100k', top_n=10, bottom_n=10)
# quick_compare('iid_group', 'dui_arrest_rate_per_100k')
# quick_map('pct_suspended', title='% Drivers on Suspended License in Fatal Crashes')

In [ ]:
# Two-variable map: bubble overlay
quick_bubble_map('ethanol_per_capita_gallons_2022', 'pct_alcohol_nhtsa_imputed',
                 title='Consumption (fill) + % Traffic Deaths from Alcohol (bubble)')

In [ ]:
# Two-variable map: bivariate 3x3 grid
quick_bivariate_map('ethanol_per_capita_gallons_2022', 'pct_alcohol_nhtsa_imputed',
                    title='Bivariate: Consumption vs % Traffic Deaths from Alcohol')

In [ ]:
# Bivariate: BAC testing rate vs fatality rate
# "States that don't test, don't know" — low testing + low reported rate = measurement artifact?
quick_bivariate_map('pct_bac_known_killed', 'alcohol_fatality_rate_per_100m_vmt',
                    title='BAC Testing Rate vs Alcohol Fatality Rate')

In [ ]:
# Bivariate: Arrest rate vs fatality rate
# "Are states catching drunk drivers or burying them?"
quick_bivariate_map('dui_arrest_rate_per_100k_reporting', 'alcohol_fatality_rate_per_100m_vmt',
                    title='DUI Arrest Rate vs Alcohol Fatality Rate')

In [ ]:
# Bivariate: Consumption vs arrest rate
# "Who's drinking and who's getting caught?"
quick_bivariate_map('ethanol_per_capita_gallons_2022', 'dui_arrest_rate_per_100k_reporting',
                    title='Alcohol Consumption vs DUI Arrest Rate')

In [ ]:
# Bivariate: Enforcement strictness (composite) vs fatality rate
# Build a quick enforcement score: sum of boolean enforcement tools
enforce_cols = ['checkpoints_permitted', 'iid_all_offender', 'has_high_bac_penalty',
                'open_container_compliant', 'pbt_authorized', 'als_alr_enacted']
# Only include columns that exist in df
available = [c for c in enforce_cols if c in df.columns]
df['enforcement_score'] = df[available].sum(axis=1)
print(f'Enforcement score range: {df["enforcement_score"].min():.0f} – {df["enforcement_score"].max():.0f}')
print(df[['state_abbr', 'enforcement_score']].sort_values('enforcement_score').head(10))

quick_bivariate_map('enforcement_score', 'alcohol_fatality_rate_per_100m_vmt',
                    title='Enforcement Strictness vs Alcohol Fatality Rate')

In [ ]:
# Bivariate map with highlighted states (always-misdemeanor outlined)
# FIPS codes for states where DUI is always a misdemeanor (felony_threshold = NULL)
always_misdemeanor_fips = {'06', '11', '13', '22', '30', '37', '42'}  # CA, DC, GA, LA, MT, NC, PA — verify with your data

# Pull actual FIPS from DuckDB for accuracy
import duckdb as _ddb
_con = _ddb.connect('data/project.duckdb', read_only=True)
always_misdemeanor_fips = set(
    _con.sql("SELECT state_fips FROM dui_criminal_status_clean WHERE felony_threshold IS NULL")
    .df()['state_fips'].str.zfill(2)
)
_con.close()
print(f'Always-misdemeanor states ({len(always_misdemeanor_fips)}): {sorted(always_misdemeanor_fips)}')

quick_bivariate_map(
    'ethanol_per_capita_gallons_2022', 'pct_alcohol_nhtsa_imputed',
    title='Consumption vs % Deaths from Alcohol (misdemeanor-only states outlined)',
    highlight_fips=always_misdemeanor_fips,
    highlight_label='DUI always a misdemeanor',
    highlight_style={'edgecolor': 'black', 'linewidth': 2.5, 'linestyle': '--'}
)

## Felony Threshold — # of Offenses Before DUI Becomes a Felony

In [ ]:
# Felony threshold grouping — uses NASID enforcement data (via export)
# NASID tracks the standard # of offenses before DUI becomes a felony.
# This supersedes the old NCSL-based coding which mixed 'first offense can be felony'
# (aggravating circumstances) with standard escalation thresholds.

def felony_label(row):
    if row['has_felony_dui'] == 0: return 'Always a misdemeanor'
    t = row['felony_dui_threshold']
    if pd.isna(t): return 'Always a misdemeanor'
    t = int(t)
    if t == 2: return 'Felony on 2nd'
    elif t == 3: return 'Felony on 3rd'
    elif t == 4: return 'Felony on 4th'
    return f'Felony on {t}th'

felony_df = df[['state_fips', 'state_abbr', 'state_name', 'has_felony_dui', 'felony_dui_threshold']].copy()
felony_df['felony_at'] = felony_df.apply(felony_label, axis=1)
print(felony_df['felony_at'].value_counts())
felony_df[['state_abbr', 'felony_at']].sort_values('state_abbr')


In [ ]:
# --- Bar chart: count of states by felony threshold ---
import matplotlib.pyplot as plt
from src.viz import PALETTE, STYLE
order = ['Always a misdemeanor', 'Felony on 4th', 'Felony on 3rd', 'Felony on 2nd']
counts = felony_df['felony_at'].value_counts().reindex(order).fillna(0).astype(int)
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#D9D9D9', '#264653', '#2A9D8F', '#E76F51']
bars = ax.barh(counts.index, counts.values, color=colors, edgecolor='white', linewidth=0.5)
# Value labels
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val} states', va='center', fontsize=11)
ax.set_xlabel('')
ax.set_title('How Many Offenses Before DUI Becomes a Felony?', fontsize=14, fontweight='bold', loc='left')
ax.set_xlim(0, counts.max() + 5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(left=False)
fig.text(0.01, 0.01, 'Source: NASID state enforcement laws, corroborated with NCSL | @unwelcomedata', fontsize=8, color='gray')
plt.tight_layout()
fig


In [ ]:
# --- Map: felony threshold by state (categorical choropleth) ---
category_colors = {
    'Felony on 2nd': '#E76F51',
    'Felony on 3rd': '#2A9D8F',
    'Felony on 4th': '#264653',
    'Always a misdemeanor': '#D9D9D9',
}
fig = choropleth_map(
    felony_df,
    column='felony_at',
    title='When Does a DUI Become a Felony?',
    subtitle='Number of DUI convictions that triggers a felony charge',
    source='NASID state enforcement laws, corroborated with NCSL',
    mode='category',
    category_colors=category_colors,
    legend_title='Becomes felony at:',
    preset='twitter_landscape',
)
fig


## First-Time vs Repeat Offenders in Fatal Crashes

**Key finding:** ~93% of impaired drivers in fatal crashes had NO prior DWI conviction.  
This aligns with NHTSA's published figure of 7% repeat-offender involvement in fatal crashes (NCSA, 2023).  
The commonly-cited '1/3 are repeat offenders' refers to *arrests*, not fatal crashes.

**Implication:** Felony escalation for repeat offenders targets only ~7% of the fatal-crash problem.

In [ ]:
# Stacked bar: first-time vs repeat, grouped by felony threshold category
prior_dwi = con.sql('''
    SELECT p.state_fips, p.pct_impaired_with_prior_dwi,
           p.impaired_drivers_known_history, p.impaired_with_prior_dwi
    FROM fars_prior_dwi_speed p
''').df()
prior_dwi['state_fips'] = prior_dwi['state_fips'].astype(str).str.zfill(2)
# Merge with felony groups
prior_merged = prior_dwi.merge(felony_df[['state_fips', 'felony_at']], on='state_fips', how='left')
# Aggregate by felony group
order = ['Always a misdemeanor', 'Felony on 4th', 'Felony on 3rd', 'Felony on 2nd']
group_agg = prior_merged.groupby('felony_at').agg(
    total_impaired=('impaired_drivers_known_history', 'sum'),
    total_with_prior=('impaired_with_prior_dwi', 'sum'),
).reindex(order)
group_agg['pct_repeat'] = (group_agg['total_with_prior'] / group_agg['total_impaired'] * 100).round(1)
group_agg['pct_first_time'] = 100 - group_agg['pct_repeat']
print('Prior DWI rates by felony threshold group:')
print(group_agg)
print()
print(f'National: {group_agg["total_with_prior"].sum()} / {group_agg["total_impaired"].sum()} = '
      f'{group_agg["total_with_prior"].sum() / group_agg["total_impaired"].sum() * 100:.1f}% repeat')


In [ ]:
# --- Stacked horizontal bar chart ---
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(12, 5))

groups = group_agg.index.tolist()
y = np.arange(len(groups))

# First-timers (left, dominant)
bars1 = ax.barh(y, group_agg['pct_first_time'], color='#264653', edgecolor='white', label='No prior DWI conviction')
# Repeat offenders (right, small)
bars2 = ax.barh(y, group_agg['pct_repeat'], left=group_agg['pct_first_time'], color='#E76F51', edgecolor='white', label='Had prior DWI conviction')

# Labels
for i, (first, repeat) in enumerate(zip(group_agg['pct_first_time'], group_agg['pct_repeat'])):
    ax.text(first/2, i, f'{first:.0f}%', ha='center', va='center', fontsize=11, color='white', fontweight='bold')
    if repeat > 3:  # Only label if visible
        ax.text(first + repeat/2, i, f'{repeat:.0f}%', ha='center', va='center', fontsize=10, color='white', fontweight='bold')

ax.set_yticks(y)
ax.set_yticklabels(groups, fontsize=11)
ax.set_xlim(0, 100)
ax.set_xlabel('')
ax.set_title('Impaired Drivers in Fatal Crashes: First-Timers vs Repeat Offenders',
             fontsize=13, fontweight='bold', loc='left')
ax.legend(loc='lower right', fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

fig.text(0.01, 0.01,
    'FARS 2024 PREV_DWI field | Validated against NHTSA NCSA 2023 (7% national rate) | @unwelcomedata',
    fontsize=8, color='gray')
plt.tight_layout()
fig

## Penalty Maps — Suspension, Jail, Fines

Using `dui_penalties_numeric` (parsed from string penalty data in 02-clean).

In [ ]:
# Load penalty data (from parquet — table created in 02-clean)
# NOTE: dui_penalties_numeric.parquet has not been created yet.
# Penalty data in DuckDB is still string-format. Skip this section until parsed.
import warnings
try:
    penalties = pd.read_parquet('data/interim/dui_penalties_numeric.parquet')
    penalties['state_fips'] = penalties['state_fips'].astype(str).str.zfill(2)
except FileNotFoundError:
    warnings.warn('dui_penalties_numeric.parquet not found — penalty maps skipped. Run penalty parsing first.')
    penalties = None

# Helper: format days as human-readable duration
def fmt_days(d):
    if pd.isna(d) or d is None:
        return ''
    d = int(d)
    if d == 0:
        return 'None'
    elif d >= 365:
        yrs = d / 365
        return f'{yrs:.1f} yr' if yrs != int(yrs) else f'{int(yrs)} yr'
    elif d >= 30 and d % 30 == 0:
        return f'{d // 30} mo'
    else:
        return f'{d}d'

# Merge with master for mapping
if penalties is not None:
    pen_map = df[['state_fips', 'state_abbr', 'state_name']].merge(
        penalties[['state_fips', 'jail_days_min', 'jail_days_max', 'suspension_days_min',
                   'suspension_days_max', 'fine_min_usd', 'fine_max_usd']],
        on='state_fips', how='left'
    )
    print(f'{len(pen_map)} states loaded')
    print(pen_map[['state_abbr', 'suspension_days_max', 'jail_days_max', 'fine_max_usd']].describe())
else:
    pen_map = None
    print('Penalty data not available — skipping penalty maps')

In [ ]:
# Map: License suspension length (max days, first offense)
if pen_map is not None:
    choropleth_map(
        pen_map,
        column='suspension_days_max',
        title='First-Offense DUI: License Suspension Length',
        subtitle='Maximum suspension days for a standard first DUI offense',
        source='State DUI penalty statutes (compiled from multiple sources)',
        mode='heat',
        cmap=['#DBEAFE', '#60A5FA', '#2563EB', '#7C3AED', '#4C1D95'],
        legend_title='Days suspended',
        preset='twitter_landscape',
        annotate=True,
    )

In [ ]:
# Map: Maximum jail time (first offense)
if pen_map is not None:
    choropleth_map(
        pen_map,
        column='jail_days_max',
        title='First-Offense DUI: Maximum Jail Time',
        subtitle='Maximum jail sentence in days for a standard first DUI offense',
        source='State DUI penalty statutes (compiled from multiple sources)',
        mode='heat',
        cmap=['#FEF3C7', '#FBBF24', '#D97706', '#DC2626', '#7F1D1D'],
        legend_title='Days (max)',
        preset='twitter_landscape',
        annotate=True,
    )

In [ ]:
# Map: Maximum fine (first offense)
if pen_map is not None:
    choropleth_map(
        pen_map,
        column='fine_max_usd',
        title='First-Offense DUI: Maximum Fine',
        subtitle='Maximum fine amount (USD) for a standard first DUI offense',
        source='State DUI penalty statutes (compiled from multiple sources)',
        mode='heat',
        cmap=['#ECFDF5', '#6EE7B7', '#10B981', '#047857', '#064E3B'],
        legend_title='Max fine ($)',
        preset='twitter_landscape',
        annotate=True,
    )

In [ ]:
# Ranked bars: top/bottom states by suspension length
if pen_map is not None:
    pen_sorted = pen_map.dropna(subset=['suspension_days_max']).sort_values('suspension_days_max', ascending=False)

    fig, ax = plt.subplots(figsize=(10, 8))
    top10 = pen_sorted.head(10)
    bot10 = pen_sorted.tail(10)
    show = pd.concat([top10, bot10])

    colors = ['#4C1D95'] * 10 + ['#DBEAFE'] * 10
    ax.barh(range(len(show)), show['suspension_days_max'], color=colors, edgecolor='white')
    ax.set_yticks(range(len(show)))
    ax.set_yticklabels(show['state_abbr'])
    ax.invert_yaxis()
    ax.set_title('License Suspension: Longest vs Shortest (First Offense)', fontsize=13, fontweight='bold', loc='left')
    ax.set_xlabel('Days')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    for i, (_, row) in enumerate(show.iterrows()):
        ax.text(row['suspension_days_max'] + 5, i, fmt_days(row['suspension_days_max']), va='center', fontsize=9)

    fig.text(0.01, 0.01, '@unwelcomedata', fontsize=8, color='gray')
    plt.tight_layout()
    fig

## Felony Threshold — Group Profiles

Compare average metrics across felony-threshold groups to see if stricter escalation
correlates with different outcomes, enforcement, or demographics.

In [ ]:
# Build a merged profile dataset: felony group + key metrics from master table
profile = df.merge(felony_df[['state_fips', 'felony_at']], on='state_fips', how='left')
# Metrics to compare across groups
metrics = {
    'pct_alcohol_nhtsa_imputed': '% traffic deaths from alcohol',
    'alcohol_fatality_rate_per_100m_vmt': 'Fatality rate per 100M VMT',
    'dui_arrest_rate_per_100k_reporting': 'DUI arrest rate per 100k',
    'ethanol_per_capita_gallons_2022': 'Per capita ethanol (gal)',
    'pct_impaired_with_prior_dwi': '% impaired with prior DWI',
    'pct_bac_known_killed': '% killed drivers BAC tested',
}
# Group means
group_order = ['Always a misdemeanor', 'Felony on 4th', 'Felony on 3rd', 'Felony on 2nd']
group_profiles = profile.groupby('felony_at')[list(metrics.keys())].mean().reindex(group_order)
group_profiles.columns = [metrics[c] for c in group_profiles.columns]
print('Group means:')
group_profiles.round(2)


In [ ]:
# --- Small multiples: one subplot per metric, raw values ---
import matplotlib.pyplot as plt
import numpy as np

# Drop '5th offense' (N=1, Wisconsin only)
plot_profiles = group_profiles.drop('5th offense', errors='ignore')
plot_order = [g for g in group_order if g != '5th offense']

n_metrics = len(plot_profiles.columns)
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

bar_colors = {'Always a misdemeanor': '#2563EB', 'Felony on 4th': '#F97316',
              'Felony on 3rd': '#DC2626', 'Felony on 2nd': '#7F1D1D'}

for i, col in enumerate(plot_profiles.columns):
    ax = axes[i]
    vals = plot_profiles[col]
    colors = [bar_colors[g] for g in plot_order]
    bars = ax.barh(plot_order, vals, color=colors, edgecolor='white', linewidth=0.5)
    
    # Value labels
    for bar, val in zip(bars, vals):
        ax.text(bar.get_width() + (vals.max() * 0.02), bar.get_y() + bar.get_height()/2,
                f'{val:.1f}', va='center', fontsize=9)
    
    ax.set_title(col, fontsize=10, fontweight='bold', loc='left')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(left=False)
    ax.set_xlim(0, vals.max() * 1.25)

# Hide unused subplots if any
for j in range(n_metrics, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Felony Threshold Group Profiles — Raw Averages', fontsize=14, fontweight='bold', x=0.01, ha='left')
fig.text(0.01, 0.01, 'Group means (N: 2nd=20, 3rd=15, 4th=8, Never=7) | 5th offense excluded (WI only) | @unwelcomedata', fontsize=8, color='gray')
plt.tight_layout()
fig

In [ ]:
# --- Heatmap version: easier to scan ---
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(12, 4))

# Normalize for color only
norm_data = plot_profiles.copy()
for col in norm_data.columns:
    col_min = norm_data[col].min()
    col_max = norm_data[col].max()
    if col_max > col_min:
        norm_data[col] = (norm_data[col] - col_min) / (col_max - col_min)
    else:
        norm_data[col] = 0.5

data_matrix = norm_data.values
im = ax.imshow(data_matrix, cmap='RdYlGn_r', aspect='auto', vmin=0, vmax=1)

# Labels
ax.set_yticks(range(len(plot_order)))
ax.set_yticklabels(plot_order, fontsize=11)
ax.set_xticks(range(len(plot_profiles.columns)))
ax.set_xticklabels(plot_profiles.columns, fontsize=9, rotation=30, ha='right')

# Annotate cells with actual (non-normalized) values
for i in range(len(plot_order)):
    for j in range(len(plot_profiles.columns)):
        raw_val = plot_profiles.iloc[i, j]
        text_color = 'white' if data_matrix[i, j] > 0.65 else 'black'
        ax.text(j, i, f'{raw_val:.1f}', ha='center', va='center', fontsize=9, color=text_color)

ax.set_title('Felony Threshold Group Profiles — Average Metrics by Group', fontsize=13, fontweight='bold', loc='left')
fig.colorbar(im, ax=ax, shrink=0.8, label='Relative (0=low, 1=high)')

fig.text(0.01, 0.01, 'Cell values are group means (raw units) | Color = normalized rank across groups | @unwelcomedata', fontsize=8, color='gray')
plt.tight_layout()
fig

## Speed Limit vs DUI Fatality Rate — Bivariate Choropleth

Do states with higher speed limits have more alcohol-impaired fatalities per VMT?  
Speed limit is a structural factor — higher speeds mean less reaction time and more lethal crashes.  
This bivariate map shows both dimensions simultaneously.

In [ ]:
quick_bivariate_map(
    'max_speed_limit_mph',
    'alcohol_fatality_rate_per_100m_vmt',
    title='Max Speed Limit vs Alcohol Fatality Rate'
)

In [ ]:
# --- Scatter view for same relationship ---
quick_scatter(
    'max_speed_limit_mph',
    'alcohol_fatality_rate_per_100m_vmt',
    title='Max Speed Limit vs DUI Fatality Rate',
    xlabel='State max speed limit (mph)',
    ylabel='Alcohol fatalities per 100M VMT',
)

In [ ]:
# --- Summary stats by speed-limit tercile ---
speed_data = df[['state_abbr', 'max_speed_limit_mph', 'alcohol_fatality_rate_per_100m_vmt']].dropna().copy()
speed_data['speed_tercile'] = pd.qcut(speed_data['max_speed_limit_mph'], 3, labels=['Low (55-65)', 'Mid (70)', 'High (75-85)'], duplicates='drop')
print('Fatality rate by speed-limit tercile:')
print(speed_data.groupby('speed_tercile')['alcohol_fatality_rate_per_100m_vmt'].agg(['mean', 'median', 'count']).round(2))
print()
print('States in each tercile:')
for t in ['Low (55-65)', 'Mid (70)', 'High (75-85)']:
    states = speed_data[speed_data['speed_tercile'] == t]['state_abbr'].sort_values().tolist()
    print(f'  {t}: {len(states)} states — {", ".join(states)}')

## Felony Lookback × Felony Threshold — Bivariate Choropleth

Two dimensions of how states treat DUI escalation:  
- **Felony threshold:** how many offenses before DUI becomes a felony (2nd, 3rd, 4th, or never)  
- **Felony lookback:** how long prior convictions count toward that threshold (5 years to lifetime)  

A state with a short lookback + high threshold is effectively saying: we forgive.  
A state with lifetime lookback + low threshold: once a DUI offender, always on notice.

In [ ]:
# --- Bivariate: Felony lookback × Felony threshold ---
# Bin lookback into 3 categories: short (5-7), standard (10-15), lifetime (99)
biv_data = df[['state_abbr', 'lookback_years', 'has_felony_dui', 'felony_dui_threshold']].copy()
biv_data = biv_data[biv_data['lookback_years'] > 0].copy()  # exclude DC, MD, NJ (no felony law)

def lookback_bin(yrs):
    if yrs <= 7: return 'Short (5–7 yr)'
    elif yrs <= 15: return 'Standard (10–15 yr)'
    else: return 'Lifetime'

def threshold_label(t):
    if pd.isna(t): return 'No felony'
    t = int(t)
    if t == 2: return 'Felony on 2nd'
    elif t == 3: return 'Felony on 3rd'
    elif t == 4: return 'Felony on 4th'
    return f'Felony on {t}th'

biv_data['lookback_cat'] = biv_data['lookback_years'].apply(lookback_bin)
biv_data['threshold_cat'] = biv_data['felony_dui_threshold'].apply(threshold_label)

# Crosstab
ct = pd.crosstab(biv_data['lookback_cat'], biv_data['threshold_cat'])
ct = ct.reindex(index=['Short (5–7 yr)', 'Standard (10–15 yr)', 'Lifetime'],
                columns=['Felony on 2nd', 'Felony on 3rd', 'Felony on 4th'])
print('Crosstab: lookback × threshold (n=48 states with felony DUI):')
print(ct)
print()
print('States by category:')
for lb in ['Short (5–7 yr)', 'Standard (10–15 yr)', 'Lifetime']:
    for th in ['Felony on 2nd', 'Felony on 3rd', 'Felony on 4th']:
        states = biv_data[(biv_data['lookback_cat']==lb) & (biv_data['threshold_cat']==th)]['state_abbr'].tolist()
        if states:
            print(f'  {lb} + {th}: {states}')

In [ ]:
# --- Bivariate choropleth: lookback × threshold ---
quick_bivariate_map(
    'lookback_years',
    'felony_dui_threshold',
    title='Felony Lookback Period vs Felony Threshold by State'
)

In [ ]:
# --- Categorical choropleth: lookback alone ---
biv_data_map = df[['state_fips', 'state_abbr', 'lookback_years']].copy()
biv_data_map['lookback_cat'] = biv_data_map['lookback_years'].apply(
    lambda x: 'No felony law' if x == 0 else lookback_bin(x))

cat_colors = {
    'Short (5–7 yr)': '#E76F51',
    'Standard (10–15 yr)': '#2A9D8F',
    'Lifetime': '#264653',
    'No felony law': '#D9D9D9',
}

fig = choropleth_map(
    biv_data_map,
    column='lookback_cat',
    title='How Long Is the Window to Trigger a DUI Felony?',
    subtitle='Lookback period: how long prior convictions count toward felony escalation',
    source='ailawyer.pro, NASID, state statutes (compiled 2026)',
    mode='category',
    category_colors=cat_colors,
    legend_title='Felony lookback',
    preset='twitter_landscape',
)
save_chart(fig, cfg, 'map_felony_lookback', preset='twitter_landscape', add_watermark='@unwelcomedata', close=False)
fig

## Felony Threshold × Checkpoint Legality

How strictly does your state treat DUI? Two independent axes:  
- **Felony threshold:** strict = felony on 2nd or 3rd; lenient = 4th or never  
- **Checkpoints:** allowed (39 states) or banned (12 states)  

States that are strict on one axis aren't necessarily strict on the other.

In [ ]:
# --- Strictness quadrants: felony threshold × checkpoint legality ---
def strictness_quad(row):
    strict_felony = row['felony_dui_threshold'] <= 3 if row['has_felony_dui'] == 1 else False
    has_checkpoint = row['checkpoints_permitted'] == 1
    if strict_felony and has_checkpoint:
        return 'Strict on both'
    elif strict_felony and not has_checkpoint:
        return 'Strict felony, no checkpoints'
    elif not strict_felony and has_checkpoint:
        return 'Lenient felony, has checkpoints'
    else:
        return 'Lenient on both'

df['strictness_quad'] = df.apply(strictness_quad, axis=1)

quad_colors = {
    'Strict on both': '#264653',
    'Strict felony, no checkpoints': '#E76F51',
    'Lenient felony, has checkpoints': '#2A9D8F',
    'Lenient on both': '#D9D9D9',
}

# Print quadrants
print('=== DUI Strictness Quadrants ===')
for cat in ['Strict on both', 'Strict felony, no checkpoints', 'Lenient felony, has checkpoints', 'Lenient on both']:
    states = df[df['strictness_quad'] == cat]['state_abbr'].sort_values().tolist()
    print(f'{cat} ({len(states)}): {", ".join(states)}')
print()

fig = choropleth_map(
    df,
    column='strictness_quad',
    title='How Strictly Does Your State Treat DUI?',
    subtitle='Strict felony = 2nd/3rd offense. Checkpoints = sobriety checkpoint legality.',
    source='NASID, NCSL, IIHS/GHSA enforcement data',
    mode='category',
    category_colors=quad_colors,
    legend_title='Strictness quadrant',
    preset='twitter_landscape',
)
save_chart(fig, cfg, 'map_strictness_quadrants', preset='twitter_landscape', add_watermark='@unwelcomedata', close=False)
fig


In [ ]:
# --- Bivariate matrix choropleth: felony threshold × checkpoints ---
# 2x2 grid: X = checkpoints (no/yes), Y = felony strictness (lenient/strict)
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from src.viz import PRESETS, STYLE, _fig_for_preset, _shift_alaska_hawaii, _load_states_geo

# Assign bivariate color
# Grid: rows = felony strictness (strict top, lenient bottom)
#        cols = checkpoints (no left, yes right)
BIV_COLORS = {
    # (strict, no checkpoint), (strict, has checkpoint)
    # (lenient, no checkpoint), (lenient, has checkpoint)
    ('Strict', 'No checkpoints'): '#E76F51',   # strict felony but no enforcement tool (coral)
    ('Strict', 'Checkpoints'): '#264653',       # strict on both (dark blue)
    ('Lenient', 'No checkpoints'): '#D9D9D9',   # lenient on both (gray)
    ('Lenient', 'Checkpoints'): '#2A9D8F',      # lenient felony but has enforcement (teal)
}

def biv_assign(row):
    strict = row['felony_dui_threshold'] <= 3 if row['has_felony_dui'] == 1 else False
    has_cp = row['checkpoints_permitted'] == 1
    felony_cat = 'Strict' if strict else 'Lenient'
    cp_cat = 'Checkpoints' if has_cp else 'No checkpoints'
    return BIV_COLORS[(felony_cat, cp_cat)]

df['biv_strict_color'] = df.apply(biv_assign, axis=1)

# Load geo
geo = _load_states_geo()
geo = _shift_alaska_hawaii(geo)
geo = geo.merge(df[['state_fips', 'biv_strict_color']].assign(
    state_fips=df['state_fips'].astype(str).str.zfill(2)),
    left_on='STATEFP', right_on='state_fips', how='left')
geo['biv_strict_color'] = geo['biv_strict_color'].fillna('#E5E7EB')

fig, ax = _fig_for_preset('twitter_landscape')
ax.set_axis_off()
for color in geo['biv_strict_color'].unique():
    geo[geo['biv_strict_color'] == color].plot(ax=ax, color=color, edgecolor='white', linewidth=0.5)

# 2x2 matrix legend
legend_ax = fig.add_axes([0.80, 0.12, 0.14, 0.14])
grid = [
    # row 0 (bottom = lenient), row 1 (top = strict)
    [BIV_COLORS[('Lenient', 'No checkpoints')], BIV_COLORS[('Lenient', 'Checkpoints')]],
    [BIV_COLORS[('Strict', 'No checkpoints')], BIV_COLORS[('Strict', 'Checkpoints')]],
]
for yi in range(2):
    for xi in range(2):
        legend_ax.add_patch(mpatches.Rectangle((xi, yi), 1, 1, color=grid[yi][xi]))

legend_ax.set_xlim(0, 2)
legend_ax.set_ylim(0, 2)
legend_ax.set_xticks([0.5, 1.5])
legend_ax.set_xticklabels(['No\ncheckpoints', 'Checkpoints\nallowed'], fontsize=7, ha='center')
legend_ax.set_yticks([0.5, 1.5])
legend_ax.set_yticklabels(['Lenient\nfelony', 'Strict\nfelony'], fontsize=7, va='center')
legend_ax.tick_params(length=0)
for spine in legend_ax.spines.values():
    spine.set_visible(False)

# Counts in each cell
counts = df['strictness_quad'].value_counts()
count_map = {
    (0, 0): counts.get('Lenient on both', 0),
    (1, 0): counts.get('Lenient felony, has checkpoints', 0),
    (0, 1): counts.get('Strict felony, no checkpoints', 0),
    (1, 1): counts.get('Strict on both', 0),
}
for (xi, yi), cnt in count_map.items():
    legend_ax.text(xi + 0.5, yi + 0.5, str(cnt), ha='center', va='center',
                   fontsize=11, fontweight='bold', color='white' if grid[yi][xi] in ['#264653','#E76F51'] else '#374151')

ax.set_title('How Strictly Does Your State Treat DUI?', fontsize=14, fontweight='bold', loc='left')
ax.text(0.01, -0.02, 'Strict felony = 2nd/3rd offense triggers felony | Checkpoints = sobriety checkpoints legal',
        transform=ax.transAxes, fontsize=9, color='gray')
ax.text(0.01, -0.05, 'Source: NASID, NCSL, IIHS/GHSA | @unwelcomedata',
        transform=ax.transAxes, fontsize=8, color='gray')

fig.subplots_adjust(left=0.02, right=0.95, top=0.90, bottom=0.08)
save_chart(fig, cfg, 'bivariate_strictness_matrix', preset='twitter_landscape', add_watermark='@unwelcomedata', close=False)
fig


In [ ]:
# --- Cleanup ---
con.close()